In [1]:
import numpy as np

# 1. Softmax 함수 (수치적 안정성 적용)
def softmax(z):
    # 각 샘플(행)별 최댓값을 빼서 지수 오버플로우 방지
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# 2. 범주형 교차 엔트로피 (Categorical Cross-Entropy) 손실 함수
def categorical_cross_entropy(y_pred, y_true):
    # log(0)으로 인한 언더플로우 방지 (1e-15 처리)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.sum(y_true * np.log(y_pred)) / y_pred.shape[0]

# 은닉층용 Sigmoid 및 미분
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)


# 3. 다중 클래스 분류 신경망 클래스
class MultiClassNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.5):
        self.learning_rate = learning_rate
        np.random.seed(42)

        # 가중치 초기화
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        """순전파"""
        # 은닉층 (Sigmoid)
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)

        # 출력층 (Softmax)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)

        return self.a2

    def backward(self, X, y_one_hot):
        """역전파"""
        m = X.shape[0]

        # [출력층 오차 계산]
        # Softmax + Cross-Entropy의 미분 결과: (예측값 - 실제 원핫 라벨)
        delta2 = (self.a2 - y_one_hot)

        # W2, b2 기울기
        dW2 = np.dot(self.a1.T, delta2) / m
        db2 = np.sum(delta2, axis=0, keepdims=True) / m

        # [은닉층 오차 계산]
        delta1 = np.dot(delta2, self.W2.T) * sigmoid_derivative(self.a1)

        # W1, b1 기울기
        dW1 = np.dot(X.T, delta1) / m
        db1 = np.sum(delta1, axis=0, keepdims=True) / m

        # [경사하강법 가중치 업데이트]
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1

    def train(self, X, y_one_hot, epochs=3000):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y_one_hot)

            if (epoch + 1) % 500 == 0:
                loss = categorical_cross_entropy(output, y_one_hot)
                # 예측 클래스(argmax)와 정답 클래스 비교하여 정확도 계산
                acc = np.mean(np.argmax(output, axis=1) == np.argmax(y_one_hot, axis=1))
                print(f"Epoch {epoch + 1:4d} | Loss: {loss:.4f} | Accuracy: {acc*100:.1f}%")

# -------------------------------------------------------------
# 4. 데이터셋 준비 (3개 클래스 분류 예제)
# 입력: 2차원 좌표 데이터 / 출력: 0, 1, 2 중 하나
X = np.array([
    [1.0, 2.0], [1.5, 1.8], [0.8, 2.5],  # Class 0
    [8.0, 8.0], [7.5, 9.0], [8.5, 7.8],  # Class 1
    [1.0, 8.0], [1.2, 9.0], [0.5, 8.5]   # Class 2
])

# 타겟 정답 (원-핫 인코딩 적용)
# Class 0: [1, 0, 0], Class 1: [0, 1, 0], Class 2: [0, 0, 1]
y_one_hot = np.array([
    [1, 0, 0], [1, 0, 0], [1, 0, 0],
    [0, 1, 0], [0, 1, 0], [0, 1, 0],
    [0, 0, 1], [0, 0, 1], [0, 0, 1]
])

# 5. 모델 생성 및 학습 (입력 2 -> 은닉 5 -> 출력 3)
model = MultiClassNeuralNetwork(input_size=2, hidden_size=5, output_size=3, learning_rate=1.0)

print("=== 학습 시작 ===")
model.train(X, y_one_hot, epochs=3000)

print("\n=== 테스트 샘플 예측 ===")
test_sample = np.array([[1.2, 2.1], [8.1, 8.2], [0.9, 8.8]])  # 각 클래스 근처의 점들
probs = model.forward(test_sample)

for i, p in enumerate(probs):
    pred_class = np.argmax(p)
    print(f"샘플 {i+1} 확률: {np.round(p, 3)} -> 최종 예측 클래스: Class {pred_class}")

=== 학습 시작 ===
Epoch  500 | Loss: 0.0043 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.0019 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.0012 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.0009 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.0007 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.0006 | Accuracy: 100.0%

=== 테스트 샘플 예측 ===
샘플 1 확률: [1. 0. 0.] -> 최종 예측 클래스: Class 0
샘플 2 확률: [0. 1. 0.] -> 최종 예측 클래스: Class 1
샘플 3 확률: [0. 0. 1.] -> 최종 예측 클래스: Class 2


In [2]:
import numpy as np
import seaborn as sns

# iris 불러오기
df = sns.load_dataset('iris')
print(df.head())

   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa


In [ ]:
# 1. Softmax 함수 (수치적 안정성 적용)
def softmax(z):
    # 각 샘플(행)별 최댓값을 빼서 지수 오버플로우 방지
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# 2. 범주형 교차 엔트로피 (Categorical Cross-Entropy) 손실 함수
def categorical_cross_entropy(y_pred, y_true):
    # log(0)으로 인한 언더플로우 방지 (1e-15 처리)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.sum(y_true * np.log(y_pred)) / y_pred.shape[0]

# 은닉층용 Sigmoid 및 미분
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)


# 3. 다중 클래스 분류 신경망 클래스
class MultiClassNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.5):
        self.learning_rate = learning_rate
        np.random.seed(42)

        # 가중치 초기화
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        """순전파"""
        # 은닉층 (Sigmoid)
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)

        # 출력층 (Softmax)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)

        return self.a2

    def backward(self, X, y_one_hot):
        """역전파"""
        m = X.shape[0]

        # [출력층 오차 계산]
        # Softmax + Cross-Entropy의 미분 결과: (예측값 - 실제 원핫 라벨)
        delta2 = (self.a2 - y_one_hot)

        # W2, b2 기울기
        dW2 = np.dot(self.a1.T, delta2) / m
        db2 = np.sum(delta2, axis=0, keepdims=True) / m

        # [은닉층 오차 계산]
        delta1 = np.dot(delta2, self.W2.T) * sigmoid_derivative(self.a1)

        # W1, b1 기울기
        dW1 = np.dot(X.T, delta1) / m
        db1 = np.sum(delta1, axis=0, keepdims=True) / m

        # [경사하강법 가중치 업데이트]
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1

    def train(self, X, y_one_hot, epochs=3000):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y_one_hot)

            if (epoch + 1) % 500 == 0:
                loss = categorical_cross_entropy(output, y_one_hot)
                # 예측 클래스(argmax)와 정답 클래스 비교하여 정확도 계산
                acc = np.mean(np.argmax(output, axis=1) == np.argmax(y_one_hot, axis=1))
                print(f"Epoch {epoch + 1:4d} | Loss: {loss:.4f} | Accuracy: {acc*100:.1f}%")

# -------------------------------------------------------------
# 4. 데이터셋 준비 (3개 클래스 분류 예제)
# 입력: 2차원 좌표 데이터 / 출력: 0, 1, 2 중 하나
X = np.array([
    [1.0, 2.0], [1.5, 1.8], [0.8, 2.5],  # Class 0
    [8.0, 8.0], [7.5, 9.0], [8.5, 7.8],  # Class 1
    [1.0, 8.0], [1.2, 9.0], [0.5, 8.5]   # Class 2
])

# 타겟 정답 (원-핫 인코딩 적용)
# Class 0: [1, 0, 0], Class 1: [0, 1, 0], Class 2: [0, 0, 1]
y_one_hot = np.array([
    [1, 0, 0], [1, 0, 0], [1, 0, 0],
    [0, 1, 0], [0, 1, 0], [0, 1, 0],
    [0, 0, 1], [0, 0, 1], [0, 0, 1]
])

# 5. 모델 생성 및 학습 (입력 2 -> 은닉 5 -> 출력 3)
model = MultiClassNeuralNetwork(input_size=2, hidden_size=5, output_size=3, learning_rate=1.0)

print("=== 학습 시작 ===")
model.train(X, y_one_hot, epochs=3000)

print("\n=== 테스트 샘플 예측 ===")
test_sample = np.array([[1.2, 2.1], [8.1, 8.2], [0.9, 8.8]])  # 각 클래스 근처의 점들
probs = model.forward(test_sample)

for i, p in enumerate(probs):
    pred_class = np.argmax(p)
    print(f"샘플 {i+1} 확률: {np.round(p, 3)} -> 최종 예측 클래스: Class {pred_class}")